***

Preparing Workspace

***

In [ ]:
# Packages
import pandas as pd
import numpy as np
import json
import requests
import os
from functools import reduce
from tqdm import tqdm
import functools as ft
import math
pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_code    = os.path.join(path_git, 'Python Code', 'BLS')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'BLS')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')

print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0,     'Functions.py')).read())
exec(open(os.path.join(path_config , 'BLS Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
exec(open(os.path.join(path_config, 'api_key.txt')).read())
api_key = dict_api[user]

***

Preparing Imports

***

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Step 01 - Supplemental Scripts', 'Step 01a - Prepare API Request Inputs.py')).read())

***

Importing

***

Version 2 (registered API key) allows us to pull:  50 Series ID's per request, 20 years of data per request, 500 requests per day

In [ ]:

list_df_years = []
year_step = 20

# Loop through the specified range of years in step intervals
for year_range_start in range(year_start, year_end + 1, year_step):
    year_range_end = min(year_range_start + year_step - 1, year_end)
    print('')
    print('Requesting data from ' + str(year_range_start) + ' to ' + str(year_range_end))

    list_df_series = []

    # Iterate through 50 Series ID at a time
    for list_series in list_series_all:

        list_df = []

        # Set up BLS API request
        url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
        url_key = '?registrationkey={}'.format(api_key)
        headers = {'Content-type': 'application/json'}
        data = json.dumps({
                    "seriesid": list_series,
                    "startyear": year_range_start,
                    "endyear": year_range_end,
                    "registrationkey": api_key
                    })

        # API request
        response = requests.post('{}{}'.format(url, url_key), headers=headers, data=data).json()

        # Translate information from dictionary results into pandas dataframe
        for i in tqdm(list_series):
            try:
                df = pd.DataFrame.from_dict(response['Results']['series'])
                df = pd.DataFrame.from_dict(df[df['seriesID'] == i]['data'].values[0])
                df = df[['year', 'periodName', 'value']]
                df = df.rename(columns = {'value': i})
                list_df.append(df)
            except Exception as e: print(i); print(e)

        # Combine all column df's together for each set of 50 Series ID's
        df_series = ft.reduce(lambda left, right: pd.merge(left, right, on = ['year', 'periodName'], how = 'left'), list_df)
        list_df_series.append(df_series)

    # Combine all sets of 50 series ID df's
    df_years = ft.reduce(lambda left, right: pd.merge(left, right, on = ['year', 'periodName'], how = 'left'), list_df_series)
    list_df_years.append(df_years)

# Combine all data from all years together
df_bls_raw = pd.concat(list_df_years)
df_bls_raw = df_bls_raw.drop_duplicates()
df_bls_raw = df_bls_raw.reset_index(drop = True)


df_bls_raw.head()

***

Exporting

***

In [ ]:
# Export
export_title = '_'.join([indicator_name, geography, 'BLS']) + '_CHAMBER_raw.csv'
print("Exporting " + export_title + " to the following location: ")
print(path_raw)
df_bls_raw.to_csv(os.path.join(path_raw, export_title), index = False)

print('')
print('Successfully exported!')

***

Processing (optional)

***

In [ ]:
df_bls = df_bls_raw.copy()

df_bls = pd.melt(df_bls, id_vars = ['year', 'periodName'], var_name = 'seriesID', value_name = 'value')    
df_bls['date_'] = df_bls['year'].astype('str') + '-' + df_bls['periodName'].astype('str')
df_bls['date_'] = pd.to_datetime(df_bls['date_'])
df_bls['value'] = df_bls['value'].astype('float32').apply(lambda x: x*1000)
df_bls = df_bls.merge(df_series_area, on = 'seriesID')

df_bls